# 进阶实践项目参考答案 03：计算病理：原图级验证与染色稳定性

从图块分类继续进入原图级聚合、分组验证和染色变化分析。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 按原图分组划分
2. 提取颜色与纹理特征
3. 比较图块级和原图级结果
4. 完成染色扰动测试
5. 定位代表性错误区域

In [1]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix
SEED=42; OUT=Path('advanced03_results'); OUT.mkdir(exist_ok=True)
paths=list(Path('/kaggle/input').rglob('meningioma_public_morphology_tiles.npz'))+list(Path('.').rglob('meningioma_public_morphology_tiles.npz')); assert paths,'需要课程 NPZ'
d=np.load(paths[0],allow_pickle=True); images=d['images']; y=d['labels']; groups=d['source_image_ids'].astype(str); split=d['split'].astype(str)
z=images.astype('float32')/255; X=np.c_[z.mean((1,2)),z.std((1,2)),np.abs(np.diff(z,axis=1)).mean((1,2)),np.abs(np.diff(z,axis=2)).mean((1,2))]
tr=split=='train'; te=split=='test'; model=RandomForestClassifier(n_estimators=250,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1).fit(X[tr],y[tr]); prob=model.predict_proba(X[te]); pred=prob.argmax(1)
test_groups=groups[te]; group_true=[]; group_pred=[]
for g in np.unique(test_groups):
    m=test_groups==g; group_true.append(int(np.bincount(y[te][m]).argmax())); group_pred.append(int(prob[m].mean(0).argmax()))
shift=z[te].copy(); shift[...,0]=np.clip(shift[...,0]*1.08,0,1); shift[...,2]*=.92; Xs=np.c_[shift.mean((1,2)),shift.std((1,2)),np.abs(np.diff(shift,axis=1)).mean((1,2)),np.abs(np.diff(shift,axis=2)).mean((1,2))]; shifted=model.predict(Xs)
cm=confusion_matrix(y[te],pred,labels=[0,1,2]); fig,ax=plt.subplots(1,2,figsize=(8,3.5)); ax[0].imshow(cm,cmap='Blues'); ax[0].set_title('Patch confusion'); ax[1].bar(['original','stain shift'],[f1_score(y[te],pred,average='macro'),f1_score(y[te],shifted,average='macro')]); ax[1].set_ylim(0,1); ax[1].set_title('Stain robustness'); fig.tight_layout(); fig.savefig(OUT/'advanced03_summary.png',dpi=150); plt.close(fig)
result={'patch_macro_f1':float(f1_score(y[te],pred,average='macro')),'slide_macro_f1':float(f1_score(group_true,group_pred,average='macro')),'stain_shift_macro_f1':float(f1_score(y[te],shifted,average='macro')),'test_slides':len(group_true)}; (OUT/'advanced03_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); print(result)


{'patch_macro_f1': 0.7835565760938895, 'slide_macro_f1': 0.3333333333333333, 'stain_shift_macro_f1': 0.33570929419986023, 'test_slides': 2}
